# Kaggle DeepSeek OCR2 Worker

Outbound WebSocket OCR worker for the gateway `/v1/ocr` endpoint.


In [ ]:
!pip install -q websockets httpx accelerate safetensors transformers==4.46.3 tokenizers==0.20.3 PyMuPDF img2pdf einops easydict addict Pillow numpy


In [ ]:
import asyncio
import base64
import json
import mimetypes
import os
import tempfile
import time
import uuid
from pathlib import Path
from urllib.parse import urlsplit

import httpx
import torch
import websockets
from transformers import AutoModel, AutoTokenizer

EMBEDDED_RUNTIME_CONFIG = {}

def config_value(name, default=None):
    if name in os.environ and os.environ[name] != '':
        return os.environ[name]
    if name in EMBEDDED_RUNTIME_CONFIG and EMBEDDED_RUNTIME_CONFIG[name] != '':
        return EMBEDDED_RUNTIME_CONFIG[name]
    path = Path('worker_config.json')
    if path.exists():
        try:
            value = json.loads(path.read_text()).get(name)
            if value not in (None, ''):
                return value
        except Exception:
            pass
    return default

GATEWAY_WS_URL = str(config_value('GATEWAY_WS_URL', '')).strip()
WORKER_TOKEN = str(config_value('WORKER_TOKEN', '')).strip()
OWNER = str(config_value('OWNER', 'unknown')).strip() or 'unknown'
OCR_MODEL = str(config_value('OCR_MODEL', 'deepseek-ocr2')).strip()
OCR_CAPACITY = max(1, int(config_value('OCR_CAPACITY', 1)))
KEEPALIVE_LOG_SECONDS = int(config_value('KEEPALIVE_LOG_SECONDS', 60))
KAGGLE_ACCELERATOR = str(config_value('KAGGLE_ACCELERATOR', 'NvidiaTeslaT4'))
MODEL_ID = str(config_value('DEEPSEEK_OCR_MODEL_ID', 'deepseek-ai/DeepSeek-OCR-2')).strip()
PROMPT = str(config_value('DEEPSEEK_OCR_PROMPT', '<image>\n<|grounding|>Convert the document to markdown.'))
BASE_SIZE = int(config_value('DEEPSEEK_OCR_BASE_SIZE', 1024))
IMAGE_SIZE = int(config_value('DEEPSEEK_OCR_IMAGE_SIZE', 768))
CROP_MODE = str(config_value('DEEPSEEK_OCR_CROP_MODE', 'true')).lower() in {'1', 'true', 'yes', 'on'}
DTYPE_NAME = str(config_value('DEEPSEEK_OCR_DTYPE', 'float16')).lower()
HF_TOKEN = str(config_value('HF_TOKEN', '')).strip()
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

GPU_COUNT = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
if DTYPE_NAME in {'fp32', 'float32'}:
    DTYPE = torch.float32
elif DTYPE_NAME in {'bf16', 'bfloat16'}:
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float16
_ORIGINAL_MASKED_SCATTER_ = torch.Tensor.masked_scatter_
def _dtype_safe_masked_scatter_(self, mask, source):
    if source.dtype != self.dtype or source.device != self.device:
        source = source.to(device=self.device, dtype=self.dtype)
    return _ORIGINAL_MASKED_SCATTER_(self, mask, source)
torch.Tensor.masked_scatter_ = _dtype_safe_masked_scatter_
NODE_ID = f"ocr-deepseek2-{OWNER}-{uuid.uuid4().hex[:8]}"
print({
    'node_id': NODE_ID,
    'owner': OWNER,
    'model': OCR_MODEL,
    'model_id': MODEL_ID,
    'dtype': str(DTYPE),
    'gpu_names': GPU_NAMES,
    'gateway': urlsplit(GATEWAY_WS_URL).netloc,
}, flush=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
try:
    model = AutoModel.from_pretrained(
        MODEL_ID,
        _attn_implementation='flash_attention_2',
        trust_remote_code=True,
        use_safetensors=True,
    )
except Exception as exc:
    print({'event': 'flash_attention_load_failed', 'error': repr(exc)}, flush=True)
    model = AutoModel.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        use_safetensors=True,
    )
model = model.eval().cuda().to(DTYPE)
print('DeepSeek OCR2 model loaded', flush=True)

def ws_url_with_token():
    if not WORKER_TOKEN:
        return GATEWAY_WS_URL
    sep = '&' if '?' in GATEWAY_WS_URL else '?'
    return f'{GATEWAY_WS_URL}{sep}token={WORKER_TOKEN}'

def safe_suffix(filename, mime_type, default='.png'):
    if filename:
        suffix = Path(filename).suffix
        if suffix:
            return suffix[:12]
    if mime_type:
        guessed = mimetypes.guess_extension(mime_type.split(';')[0].strip())
        if guessed:
            return guessed
    return default

def decode_data_url(value):
    header, encoded = value.split(',', 1)
    mime_type = 'application/octet-stream'
    if header.startswith('data:'):
        mime_type = header[5:].split(';', 1)[0] or mime_type
    return base64.b64decode(encoded), mime_type

async def materialize_input(request, job_id):
    source = request.get('image_url') or request.get('document_url')
    encoded = request.get('image_base64') or request.get('document_base64')
    filename = request.get('filename') or f'{job_id}.png'
    mime_type = request.get('mime_type') or ''
    if source:
        if source.startswith('data:'):
            content, detected_mime = decode_data_url(source)
            mime_type = mime_type or detected_mime
        else:
            async with httpx.AsyncClient(timeout=120) as client:
                response = await client.get(source)
                response.raise_for_status()
                content = response.content
                mime_type = mime_type or response.headers.get('content-type', '')
                path_name = Path(urlsplit(source).path).name
                filename = request.get('filename') or path_name or filename
    elif encoded:
        if encoded.startswith('data:'):
            content, detected_mime = decode_data_url(encoded)
            mime_type = mime_type or detected_mime
        else:
            content = base64.b64decode(encoded)
    else:
        raise ValueError('OCR request has no input')
    suffix = safe_suffix(filename, mime_type)
    path = Path(tempfile.gettempdir()) / f'{job_id}{suffix}'
    path.write_bytes(content)
    return path, {'filename': filename, 'mime_type': mime_type, 'bytes': len(content)}

def read_output_text(output_dir):
    candidates = []
    for pattern in ('*.mmd', '*.md', '*.txt', '*.html', '*.json'):
        candidates.extend(Path(output_dir).rglob(pattern))
    for path in sorted(candidates, key=lambda item: item.stat().st_mtime, reverse=True):
        try:
            text = path.read_text(encoding='utf-8')
            if text.strip():
                return text, str(path)
        except Exception:
            continue
    return '', ''

def run_deepseek_ocr(path, request, input_meta):
    started = time.time()
    output_dir = Path(tempfile.gettempdir()) / f"deepseek-ocr2-{uuid.uuid4().hex[:8]}"
    output_dir.mkdir(parents=True, exist_ok=True)
    options = request.get('options') or {}
    prompt = str(options.get('prompt') or PROMPT)
    base_size = int(options.get('base_size') or BASE_SIZE)
    image_size = int(options.get('image_size') or IMAGE_SIZE)
    crop_mode = options.get('crop_mode')
    if crop_mode is None:
        crop_mode = CROP_MODE
    result = model.infer(
        tokenizer,
        prompt=prompt,
        image_file=str(path),
        output_path=str(output_dir),
        base_size=base_size,
        image_size=image_size,
        crop_mode=bool(crop_mode),
        save_results=True,
    )
    saved_text, saved_path = read_output_text(output_dir)
    text = saved_text or (result if isinstance(result, str) else str(result))
    return {
        'text': text,
        'markdown': text,
        'pages': [{'index': 0, 'markdown': text}],
        'data': {'raw_result': str(result), 'output_file': saved_path},
        'metadata': {
            'backend': 'deepseek-ocr2',
            'model_id': MODEL_ID,
            'prompt': prompt,
            'base_size': base_size,
            'image_size': image_size,
            'crop_mode': bool(crop_mode),
            'input': input_meta,
            'elapsed_seconds': round(time.time() - started, 3),
        },
    }

async def handle_ocr_job(ws, job):
    job_id = str(job.get('job_id') or '')
    request = job.get('request') or {}
    try:
        path, input_meta = await materialize_input(request, job_id)
        result = await asyncio.to_thread(run_deepseek_ocr, path, request, input_meta)
        await ws.send(json.dumps({'type': 'ocr_done', 'job_id': job_id, 'result': result}))
    except Exception as exc:
        await ws.send(json.dumps({'type': 'ocr_error', 'job_id': job_id, 'error': repr(exc)}))

async def heartbeat_loop(ws):
    while True:
        await asyncio.sleep(20)
        await ws.send(json.dumps({'type': 'heartbeat', 'node_id': NODE_ID}))

async def worker_loop():
    if not GATEWAY_WS_URL:
        raise RuntimeError('GATEWAY_WS_URL is required')
    while True:
        try:
            async with websockets.connect(ws_url_with_token(), ping_interval=20, ping_timeout=20, max_size=None) as ws:
                await ws.send(json.dumps({
                    'type': 'register',
                    'node_id': NODE_ID,
                    'owner': OWNER,
                    'model': OCR_MODEL,
                    'accelerator': f'{KAGGLE_ACCELERATOR}x{GPU_COUNT or 0}',
                    'capacity': OCR_CAPACITY,
                }))
                print('registered', await ws.recv(), flush=True)
                hb_task = asyncio.create_task(heartbeat_loop(ws))
                try:
                    async for raw in ws:
                        message = json.loads(raw)
                        if message.get('type') == 'terminate':
                            print({'event': 'terminate', 'reason': message.get('reason')}, flush=True)
                            return
                        if message.get('type') == 'ocr_job':
                            await handle_ocr_job(ws, message)
                finally:
                    hb_task.cancel()
        except Exception as exc:
            print('worker reconnect after error', repr(exc), flush=True)
            await asyncio.sleep(10)

await worker_loop()
